# Численное моделирование распространения сейсмических волн в двумерной среде MILEN SEM 2D. Часть вторая.

## Глава II: Расчет в CAE-Fidesys

### Научная задачаМы уже построили конечноэлементную модель для программы CAE-FidesysТеперь нужно её расчитать.

## Код для тестового расчета в Fidesys ##

Мы задаем пульс Рикера с частотой 30, амплитудой 1e10 через поровое давление, "размазанное" по элементам верхнего слоя. Для эффективного размазывания мы будем использовать функцию Гаусса с центром в точке источника на 5 м под землей и отклонением ("радиусом пятна") в 2 м: 

Перед тем, как расчитывать, собственно, полноволновую картину, мы должны подобрать корректный порядок спектрального элемента. Согласно таблице Уханова, для принятной дискретизации он составляет 4-5. Проверим на центральной части, установив там константный материал и убрав 40 нижних слоев.

Первый тест - без волн Рэлея. Вырежем из центральной области квадрат и пустим волну той же конфигурации источника по центру квадрата с расчетом, что бы она прошла по всему квадрату

create porepressure on surface 1  value 0create formula 1 'exp(-0.5*sqrt((x-4880)^2+(y-600)^2)/4)/(2.5066*4)*ricker(1e+10, 30, 0, time)'modify porepressure 1  formula 1delete block alldelete material allset duplicate block elements oncreate material 1modify material 1 name 'TestMat'modify material 1 set property 'DENSITY' value 1823.78modify material 1 set property 'BIOT_ALPHA' value 1modify material 1 set property 'MODULUS' value 8.86244e+09modify material 1 set property 'POISSON' value 0.283926block 1 add surface allblock 1 material 1 cs 1 category plane order 2analysis type dynamic elasticity dim2 planestrain preload offdynamic method full_solution scheme explicit maxsteps 10000 maxtime 0.25dynamic results everytime 0.05output nodalforce off energy off record3d off material on without_smoothing off fullperiodic offcalculation start path '/home/antonov/Документы/CAE-Fidesys-8.1/dev_2_1_milen2Do2_pore_center_simple.pvd'

Поскольку материал однородный, а сетка - не очень, значения скорости волны подобраны так, что бы она была в 2 раза быстрее минимальной (поверхностной) (удолетворяла характерному размеру элемента сетки в глубинной части модели). В обмен на это мы будем добиваться отсутствия помех 1% по нижней части модели. Примечание: таблицы Уханова составлялись для идеально ровной сетки, ровного материала и другого типа сигнала. Поэтому мы опираемся на них как на нижнюю оценку, но делаем перекалибровку.

delete block alldelete material allset duplicate block elements oncreate material 1modify material 1 name 'TestMat'modify material 1 set property 'DENSITY' value 1823.78modify material 1 set property 'BIOT_ALPHA' value 1modify material 1 set property 'MODULUS' value 8.86244e+09modify material 1 set property 'POISSON' value 0.283926block 1 add surface allblock 1 material 1 cs 1 category plane order 2create porepressure on surface 1  value 0create formula 1 'exp(-0.5*sqrt((x-5875)^2+(y-5)^2)/2)/(2.5066*2)*ricker(1e+10, 30, 0, time)'modify porepressure 1  formula 1analysis type dynamic elasticity dim2 planestrain preload offdynamic method full_solution scheme explicit maxsteps 10000 maxtime 0.5dynamic results everytime 0.05output nodalforce off energy off record3d off material on without_smoothing off fullperiodic offcalculation start path '/home/antonov/Документы/CAE-Fidesys-8.1/dev_2_1_milen2Do2_pore_center_simple.pvd'

Аналогичным образом посчитаем порядки до 5:

create porepressure on surface all  value 0create formula 1 'exp(-0.5*sqrt((x-5875)^2+(y-5)^2)/2)/(2.5066*2)*ricker(1e+10, 30, 0, time)'modify porepressure 1  formula 1block all order 2create receiver on curve 1  velocity 1 1 1create absorption on curve 29 32 113 98 44 59 71 104 41 119 122 26 50 137 65 20 161 164 167 170 173 176 131 134 14 23 53 107 80 77 86 89 95 83 5 68 125 35 56 110 140 143 146 101 149 152 128 17 116 74 155 158 11 47 62 8 2 92 38 179 182 185 188 191 194 197 200 203 206 209 212 215 218 221 224 create absorption on curve 226 create absorption on curve 3 30 33 114 60 72 105 42 120 27 51 48 138 66 21 162 165 168 171 174 177 132 135 15 54 108 84 81 78 87 96 69 123 126 36 57 111 18 141 144 99 102 147 150 153 129 117 75 156 159 12 45 63 24 9 90 6 93 39 180 183 186 189 192 195 198 201 204 207 210 213 216 219 222 225 analysis type dynamic elasticity dim2 planestrain preload offdynamic method full_solution scheme explicit maxsteps 10000 maxtime 4dynamic results everytime 0.01output nodalforce off energy off record3d off material on without_smoothing off fullperiodic off

create porepressure on node 24468 4805 28613 1859 28159 20803 1447 11506 4542 29821 6708 3091 2726 20698 11237 24960 23184 17661 28908 29542 18266 21450 11190 5994 13410 8022 11744 11300 11389 25892 5522 1968 488 24182 16571 26771 10084 16720 29823 11161 21466 970 27394 13989 3483 21546 12966 11535 6387 22333 11667 2804 29254 26404 19961 20618 25470 10820 2441 27773 28907 12608 24373 395 4376 27085 27097 1936 11343 9222 14584 1391 4475 29138 6219 3589 118 14632 10285 1911 12111 10186 11499 28440 11206 5225 22856 4862 16823 27813 19271 8824 24529 2038 13151 23663 7721 6772 16073 12653 17545 16611 29181 21737 14888 27850 25479 21917 12904 21213 6904 5760 223 21016 16074 19175 23578 26565 27211 1503 15271 10539 28377 2822 8527 16335 24279 7532 26214 20893 17640 23434 16515 9281 16259 5777 28369 10312 18883 7296 24890 18456 11293 4982 8284 6496 10437 7554 15671 13652 28401 22190 9179 28635 4116 11752 3541 2027 21455 14017 22472 5370 18698 18932 19570 28502 2100 13968 22760 26511 2790 29913 23659 14509 17342 17114 12974 21940 3800 13625 11075 5995 9046 16092 18726 23645 17529 16822 564 21122 2392 26331 13178 1800 1867 13956 13248 23995 16344 18100 18501 13513 468 4768 9274 11044 28119 16222 10180 28237 9689 16823 2285 29558 12570 28181 15166 11382 8833 16506 25055 9595 5788 26153 8622 18083 3993 2693 5389 22806 6883 23732 8168 12630 23963 13774 16548 15532 12742 5287 16111 8273 8141 10633 5553 6739 531 26072 60 2523 20860 17809 26424 29748 9629 10267 26682 25670 24109 626 28155 21156 9256 13795 18850 1457 22350 5488 10351 22904 29805 18975 9051 15352 25079 27688 13509 8296 246 14219 15781 15505 12751 13718 7316 12629 8712 10838 9018 4110 2453 13873 12874 1952 14813 create formula 1 'ricker(1e+07, 30, 0, time)'modify porepressure 1 formula 1